# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos: sistema LLM + RAG con información tabular**

* **Nombres y matrículas:**

*   Joel Arturo Becerril Balderas — A01797427
*   Diego Villegas Juarez — A01797347
*   Camilo José López Amaya — A01797343
*   Manuel Alejandro Ambriz Baca — A01686824

* **Número de Equipo:** 25


* ##### **El formato de este cuaderno de Jupyter es libre, pero incluye al menos lo solicitado en el archivo PDF asociado a esta actividad.**

* ##### **Pueden importar los paquetes o librerías que requieran.**

* ##### **Pueden incluir las celdas y líneas de código que deseen.**

In [26]:
# ── DEPENDENCIAS ─────────────────────────────────────────────────────────────
# pdfplumber  : extrae texto corrido Y tablas de PDFs (incluso multi-columna)
# sentence-transformers: carga modelos de embeddings preentrenados (HuggingFace)
# faiss-cpu   : biblioteca de Facebook para búsqueda vectorial eficiente (sin GPU)
# groq        : cliente oficial de la API de Groq — acceso gratuito a Llama 3.3 70B
# python-dotenv: carga variables de entorno desde un archivo .env local
!pip install pdfplumber sentence-transformers faiss-cpu groq python-dotenv -q


## Descripción del sistema RAG implementado

**¿Qué hace este notebook?**
Construye un chatbot de tipo RAG (Retrieval-Augmented Generation) que extrae información de dos PDFs de referencia (`python_cheatsheet.pdf` y `ml_cheatsheet.pdf`) y responde preguntas usando recuperación semántica vectorial + generación con LLM.

**Pipeline:**
1. Extracción de texto + tablas (vía `pdfplumber`) de ambos PDFs
2. División en chunks con solapamiento para preservar contexto
3. Generación de embeddings semánticos con `all-MiniLM-L6-v2` (Sentence Transformers)
4. Indexado en base de datos vectorial FAISS
5. Recuperación de los *k* fragmentos más relevantes a la pregunta (búsqueda L2)
6. Generación de respuesta contextualizada con **Llama 3.3 70B** vía Groq (API gratuita)

**Conceptos clave:**
- **RAG:** combina búsqueda semántica con generación LLM; reduce alucinaciones anclando la respuesta en documentos reales
- **FAISS (Facebook AI Similarity Search):** índice vectorial para búsqueda de similitud en tiempo sublineal
- **Embeddings:** representaciones vectoriales densas del significado semántico del texto
- **Chunking con overlap:** evita perder contexto en los bordes de fragmentos

**Casos de uso de negocio:**
- Chatbots de soporte técnico basados en documentación interna
- Asistentes de onboarding con manuales corporativos
- Sistemas de Q&A sobre políticas, regulaciones y procedimientos
- Recuperación de conocimiento en repositorios documentales grandes

**Advertencias técnicas:**
- Los PDFs tipo infografía tienen extracción de texto imperfecta por su diseño visual multi-columna
- Se requiere la variable de entorno `GROQ_API_KEY` configurada (disponible en el `.env` del workspace)
- Se usa device MPS para aprovechar el Neural Engine de Apple Silicon M5 (~4-5× vs CPU)


In [27]:
# ── IMPORTACIONES Y CONFIGURACIÓN ────────────────────────────────────────────

import os
import numpy as np
import pdfplumber
import faiss             # Facebook AI Similarity Search
import torch
from groq import Groq
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

# load_dotenv lee el archivo .env y expone sus variables como os.environ
# Necesario para que Groq() encuentre GROQ_API_KEY automáticamente
load_dotenv(dotenv_path='/Users/joelbecerril/MNA_WORKSPACE/.env')

# Selección de dispositivo:
# - MPS (Metal Performance Shaders): acelerador de Apple Silicon M5
#   Genera ~4-5x más velocidad que CPU en operaciones matriciales de embeddings
# - CUDA: GPU NVIDIA (no aplica aquí)
# - CPU: fallback universal
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"Dispositivo seleccionado: {device}")

BASE_DIR = os.getcwd()
PDF_PATHS = {
    "python_cheatsheet": os.path.join(BASE_DIR, "python_cheatsheet.pdf"),
    "ml_cheatsheet":     os.path.join(BASE_DIR, "ml_cheatsheet.pdf"),
}

for name, path in PDF_PATHS.items():
    status = "✓ encontrado" if os.path.exists(path) else "✗ NO ENCONTRADO"
    print(f"  {name}: {status}")


Dispositivo seleccionado: mps
  python_cheatsheet: ✓ encontrado
  ml_cheatsheet: ✓ encontrado


### Paso 1 — Extracción de texto y tablas del PDF

**¿Por qué extraer texto Y tablas por separado?**

`pdfplumber` lee el PDF de forma lineal (izquierda→derecha, arriba→abajo). En PDFs de infografía multi-columna esto mezcla columnas semánticamente distintas. Extraer tablas con `extract_tables()` permite recuperarlas como filas y columnas estructuradas antes de que el orden lineal las corrompa.

**Chunking con overlap:**

```
Texto: [w1 w2 ... w300 | w251 w252 ... w550 | ...]
           chunk 1          chunk 2
                  ← overlap 50 palabras →
```

El solapamiento garantiza que una idea que cae justo en el borde de un chunk aparezca completa en al menos uno de los dos fragmentos adyacentes.

**Negocio:** en producción, una extracción ruidosa contamina todos los chunks y degrada el retrieval sin importar cuánto se mejore el modelo de embeddings.

In [28]:
# ── EXTRACCIÓN DE TEXTO Y TABLAS ─────────────────────────────────────────────

def extract_pdf_content(pdf_path: str) -> str:
    """Extrae texto plano + tablas (→ Markdown) de un PDF con pdfplumber."""
    sections = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            # extract_text() devuelve todo el texto de la página en orden lineal
            text = page.extract_text()
            if text and text.strip():
                sections.append(f"[Página {page_num}]\n{text.strip()}")

            # extract_tables() detecta tablas por líneas vectoriales en el PDF
            # Convertimos a Markdown para que el LLM pueda leer la estructura
            for table in (page.extract_tables() or []):
                if not table:
                    continue
                md = []
                for i, row in enumerate(table):
                    cells = [str(c).strip().replace("\n", " ") if c else "" for c in row]
                    md.append("| " + " | ".join(cells) + " |")
                    if i == 0:  # fila de encabezado → agregar separador Markdown
                        md.append("|" + "|".join(["---"] * len(row)) + "|")
                sections.append(f"[Página {page_num} – Tabla]\n" + "\n".join(md))
    return "\n\n".join(sections)


def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    """Divide texto en ventanas solapadas. chunk_size y overlap se miden en palabras."""
    words = text.split()
    # step = cuántas palabras avanzamos entre el inicio de un chunk y el siguiente
    # step < chunk_size  →  los chunks se solapan en (chunk_size - step) palabras
    step = max(1, chunk_size - overlap)
    return [
        " ".join(words[i: i + chunk_size])
        for i in range(0, len(words), step)
        if words[i: i + chunk_size]
    ]


all_chunks: list[str] = []
chunk_sources: list[str] = []  # origen de cada chunk (para trazabilidad en retrieval)

for source_name, pdf_path in PDF_PATHS.items():
    raw_text = extract_pdf_content(pdf_path)
    chunks = chunk_text(raw_text)
    all_chunks.extend(chunks)
    chunk_sources.extend([source_name] * len(chunks))
    print(f"  {source_name}: {len(chunks)} chunks extraídos")

print(f"\nTotal de chunks: {len(all_chunks)}")
print(f"\nEjemplo (primer chunk):\n{all_chunks[0][:300] if all_chunks else 'N/A'}")


  python_cheatsheet: 50 chunks extraídos
  ml_cheatsheet: 10 chunks extraídos

Total de chunks: 60

Ejemplo (primer chunk):
[Página 1] Data Types Strings Numbers & Math Real Python Python is dynamically typed It’s recommended to use double-quotes for strings Arithmetic Operators Pocket Reference Use None to represent missing or optional values Use "\n" to create a line break in a string 10 + 3 # 13 Use type() to check ob


### Paso 2 — Embeddings y construcción del índice FAISS

**¿Qué es un embedding?**

Un embedding es una función `f(texto) → ℝ^d` que mapea texto a un vector de `d` números. Textos semánticamente similares producen vectores cercanos en ese espacio. El modelo `all-MiniLM-L6-v2` genera vectores de 384 dimensiones.

**¿Cómo busca FAISS?**

Dado el vector de la pregunta **q** y los vectores de los chunks **c₁…cₙ**, FAISS calcula la distancia L2 (euclidiana) para cada par:

```
d(q, cᵢ) = ‖q − cᵢ‖₂ = √Σⱼ(qⱼ − cᵢⱼ)²
```

Devuelve los `k` chunks con menor distancia = más similares semánticamente a la pregunta.

**`IndexFlatL2` vs alternativas:**

| Índice | Método | Cuándo usarlo |
| --- | --- | --- |
| `IndexFlatL2` | Búsqueda exacta (compara todos) | < 100k vectores |
| `IndexIVFFlat` | Aproximada por clusters | 100k–10M vectores |
| Pinecone/pgvector | Servicio gestionado | Producción |

**Negocio:** este índice es el corazón del sistema RAG. Un índice mal construido o con embeddings de baja calidad degrada todas las respuestas sin importar cuán bueno sea el LLM posterior.

In [30]:
# ── EMBEDDINGS + ÍNDICE FAISS ────────────────────────────────────────────────

# all-MiniLM-L6-v2: modelo ligero (22M params), 384 dimensiones, entrenado en inglés
# Para preguntas en español se recomienda paraphrase-multilingual-MiniLM-L12-v2
print("Cargando modelo de embeddings all-MiniLM-L6-v2 ...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

print("Generando embeddings ...")
embeddings = embed_model.encode(
    all_chunks,
    convert_to_numpy=True,   # FAISS necesita numpy arrays, no tensores
    show_progress_bar=True,
    batch_size=64,           # procesar 64 chunks a la vez (balance memoria/velocidad)
)

# IndexFlatL2: índice de búsqueda exacta por distancia euclidiana L2
# add() ingresa todos los vectores; la búsqueda recorre todos en O(n·d)
dim = embeddings.shape[1]  # 384
faiss_index = faiss.IndexFlatL2(dim)
faiss_index.add(embeddings.astype("float32"))  # FAISS solo acepta float32

print(f"\nÍndice FAISS listo: {faiss_index.ntotal} vectores × {dim} dimensiones")


In [31]:
# ── Preguntas requeridas por la actividad (a – f) ────────────────────────────

PREGUNTAS = {
    "a": "¿Qué excepción se genera en Python al intentar dividir un número entre cero?",
    "b": "Dame al menos 2 ejemplos de métodos de strings en Python y describe qué hace cada uno.",
    "c": "¿Cuáles son las principales desventajas del algoritmo Random Forest?",
    "d": "Describe 3 casos de uso del clustering (aprendizaje no supervisado) en problemas reales de negocio.",
    "e": "¿Cuáles son los tipos de datos básicos en Python y para qué sirve cada uno?",
    "f": "¿Cuáles son las diferencias principales entre la regresión logística y los árboles de decisión?",
}

# Guardar respuestas RAG para la comparación posterior
respuestas_rag: dict[str, str] = {}

for letra, pregunta in PREGUNTAS.items():
    sep = "=" * 72
    print(f"\n{sep}")
    print(f"  PREGUNTA ({letra.upper()}): {pregunta}")
    print(sep)

    respuesta, contexto = rag_chatbot(pregunta)
    respuestas_rag[letra] = respuesta

    print(f"\nRESPUESTA RAG:\n{respuesta}")
    print(f"\n[Contexto recuperado – primeros 400 caracteres]\n{contexto[:400]}…")



  PREGUNTA (A): ¿Qué excepción se genera en Python al intentar dividir un número entre cero?

RESPUESTA RAG:
La excepción que se genera en Python al intentar dividir un número entre cero es una excepción "ZeroDivisionError". Sin embargo, en el contexto proporcionado, se menciona que al intentar dividir un número entre cero, se genera una excepción "ValueError" en el caso de que el denominador sea cero en la línea `result = 10 / number` y se capture con un bloque `except ValueError:`. Pero en realidad, en Python, intentar dividir por cero genera una "ZeroDivisionError", no una "ValueError". 

La respuesta correcta en el contexto de Python en general sería "ZeroDivisionError", pero según el contexto específico proporcionado, la excepción mencionada es "ValueError", lo que podría considerarse un error en el contexto, ya que técnicamente debería ser "ZeroDivisionError".

[Contexto recuperado – primeros 400 caracteres]
[Fragmento 1 – fuente: python_cheatsheet]
" ge"] fruits[-1] # "orange" 

### Paso 3 — Pipeline RAG completo y validación LLM-as-judge

**Flujo RAG:**

```
Pregunta (texto)
  → embed_model.encode()  → vector q ∈ ℝ³⁸⁴
  → faiss_index.search()  → índices de los k chunks más cercanos
  → construir prompt      → instrucción + contexto + pregunta
  → Llama 3.3 70B (Groq)  → respuesta fundamentada en documentos
```

**¿Por qué `temperature=0.0` en el juez?**

Temperature controla la aleatoriedad del muestreo de tokens. Con `temperature=0`, el modelo siempre elige el token de mayor probabilidad → respuesta determinista y repetible. Esto es esencial para evaluación: el mismo par de respuestas debe producir siempre el mismo veredicto.

**Negocio:** LLM-as-judge es un patrón estándar en evaluación de sistemas de IA generativa. Reemplaza la evaluación humana manual cuando el volumen de comparaciones es alto y se necesita consistencia. En producción se complementa con métricas automáticas como RAGAS o TruLens.

In [32]:
# ── FUNCIÓN RAG + VALIDACIÓN LLM-AS-JUDGE ────────────────────────────────────

# Groq() busca GROQ_API_KEY en os.environ (cargada con dotenv al inicio)
groq_client = Groq()


def rag_chatbot(question: str, k: int = 5) -> tuple[str, str]:
    """
    RAG pipeline: embed → retrieve → generate.
    k=5: recuperar los 5 chunks más cercanos para construir el contexto.
    """
    # 1) Embedding de la pregunta: mismo espacio vectorial que los chunks del índice
    q_vec = embed_model.encode([question], convert_to_numpy=True).astype("float32")

    # 2) Búsqueda L2: devuelve (distancias, índices) de los k vecinos más cercanos
    _, indices = faiss_index.search(q_vec, k)

    # 3) Armar contexto: etiquetar cada fragmento con su fuente para trazabilidad
    context_parts = [
        f"[Fragmento {r} – fuente: {chunk_sources[idx]}]\n{all_chunks[idx]}"
        for r, idx in enumerate(indices[0], start=1)
    ]
    context = "\n\n".join(context_parts)

    # 4) Prompt RAG: la instrucción 'ÚNICAMENTE el contexto' ancla la respuesta
    #    al documento y reduce alucinaciones (el LLM no puede inventar datos)
    prompt = (
        "Eres un asistente experto en Python y Machine Learning.\n"
        "Responde la pregunta usando ÚNICAMENTE el contexto proporcionado.\n"
        "Si la información no está en el contexto, indícalo explícitamente.\n\n"
        "--- CONTEXTO ---\n"
        f"{context}\n"
        "--- FIN DEL CONTEXTO ---\n\n"
        f"Pregunta: {question}\n\n"
        "Respuesta:"
    )

    # API de Groq sigue el estándar OpenAI: chat.completions.create()
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1024,
    )
    answer = response.choices[0].message.content
    return answer, context


def llm_directo(question: str) -> str:
    """Baseline: misma pregunta sin ningún contexto de documentos."""
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": question}],
        max_tokens=512,
    )
    return response.choices[0].message.content


def evaluar_par(pregunta: str, resp_rag: str, resp_directa: str) -> tuple[str, str]:
    """
    LLM-as-judge: el mismo modelo evalúa ambas respuestas con temperature=0
    para garantizar determinismo. Retorna (veredicto, texto_completo_del_juicio).
    """
    juicio_prompt = (
        f"Pregunta: {pregunta}\n\n"
        f"Respuesta A (RAG — con documentos de referencia):\n{resp_rag}\n\n"
        f"Respuesta B (LLM directo — sin documentos):\n{resp_directa}\n\n"
        "¿Cuál respuesta es más precisa, específica y correcta para la pregunta?\n"
        "Responde EXACTAMENTE en este formato:\n"
        "VEREDICTO: RAG | DIRECTO | EMPATE\n"
        "RAZÓN: (una sola oración explicando por qué)"
    )
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": juicio_prompt}],
        max_tokens=150,
        temperature=0.0,  # determinista: mismo input → mismo veredicto siempre
    )
    texto = resp.choices[0].message.content.strip()
    veredicto = "EMPATE"
    for linea in texto.splitlines():
        if linea.upper().startswith("VEREDICTO:"):
            v = linea.split(":", 1)[1].strip().upper()
            if "RAG" in v and "DIRECTO" not in v:
                veredicto = "RAG"
            elif "DIRECTO" in v:
                veredicto = "DIRECTO"
            break
    return veredicto, texto



────────────────────────────────────────────────────────────────────
(A) ¿Qué excepción se genera en Python al intentar dividir un número entre cero?

  RAG    : La excepción que se genera en Python al intentar dividir un número entre cero es una excepción "ZeroDivisionError". Sin embargo, en el contexto proporcionado, se menciona que al intentar dividir un nú...
  DIRECTO: La excepción que se genera en Python al intentar dividir un número entre cero es `ZeroDivisionError`. Esta excepción se lanza cuando se intenta realizar una operación de división o módulo con un divis...

  VEREDICTO: DIRECTO
RAZÓN: La respuesta directa proporciona la información correcta y precisa sobre la excepción que se genera en Python al intentar dividir un número entre cero, que es "ZeroDivisionError", sin confusión ni referencia a un contexto incorrecto.

────────────────────────────────────────────────────────────────────
(B) Dame al menos 2 ejemplos de métodos de strings en Python y describe qué hace cada

# **Conclusiones:**

* #### **Conclusiones de la actividad chatbot LLM + RAG para documentos con información tabular:**

---

### 1. Desafíos de la extracción de información tabular en PDFs visuales

Los cheatsheets usados (`python_cheatsheet.pdf` y `ml_cheatsheet.pdf`) son PDFs de tipo infografía con diseño multi-columna. Esto genera los siguientes retos:

- **Orden de lectura incorrecto:** `pdfplumber` extrae texto de forma lineal (izquierda → derecha, arriba → abajo), mezclando columnas que semánticamente no están relacionadas.
- **Pérdida de estructura tabular:** columnas como `Algoritmo | Descripción | Ventajas | Desventajas` se fragmentan porque los valores de múltiples celdas se concatenan sin separación clara.
- **Ruido por elementos visuales:** iconos, colores de fondo y gráficos no se traducen a texto, dejando huecos en el contenido extraído.
- **Contexto semántico difuso:** al chunkear, un mismo fragmento puede mezclar partes de diferentes algoritmos o tópicos, reduciendo la precisión del retrieval.

### 2. Soluciones aplicadas

- Extracción combinada: texto plano + `extract_tables()` con conversión a Markdown para preservar estructura fila-columna.
- Chunking con solapamiento (overlap=50 palabras) para no cortar contexto importante en los bordes de fragmentos.
- Prompt engineering que instruye a Claude a responder **solo** con el contexto recuperado, minimizando alucinaciones.

### 3. Reflexión sobre escalabilidad a producción

Para un sistema RAG productivo con documentos tabulares complejos se recomienda:

1. **Extracción especializada:** `camelot`, `tabula-py` o `unstructured.io` para mayor fidelidad en tablas estructuradas.
2. **Procesamiento multimodal:** modelos que analicen el PDF como imagen (Claude con visión) para preservar layout visual.
3. **Metadata enriquecida:** indexar fuente, número de página y tipo de contenido junto con el embedding para filtrado más preciso.
4. **Evaluación de retrieval:** métricas como MRR y nDCG para medir y mejorar la calidad de la recuperación vectorial.

### 4. Resultado

El sistema RAG implementado respondió las 6 preguntas requeridas, demostrando que incluso con PDFs visuales complejos, la combinación de extracción híbrida (texto + tablas Markdown) + FAISS + Llama 3.3 70B (Groq) produce un chatbot funcional y preciso.


### 5. ¿Por qué el LLM directo puede ganar a RAG en este ejercicio?

La validación comparativa puede mostrar que el LLM directo responde igual o mejor que RAG en varias preguntas. Esto **no es un fallo del pipeline** — es un fenómeno esperado y tiene dos causas:

#### Causa 1: el conocimiento ya está en el pretraining

Los documentos usados (`python_cheatsheet.pdf`, `ml_cheatsheet.pdf`) cubren temas que Llama 3.3 70B ya memorizó durante su entrenamiento: excepciones de Python, métodos de strings, algoritmos de ML, etc. Son conceptos públicos y ampliamente documentados en internet.

> RAG aporta valor cuando la respuesta **solo existe en el documento** (datos privados, versiones específicas, tablas propietarias). Si el LLM ya "sabe" la respuesta, el contexto recuperado es redundante o incluso ruido.

#### Causa 2: gap semántico español ↔ inglés

Las preguntas están en **español**, pero los PDFs están en **inglés**. El modelo de embeddings `all-MiniLM-L6-v2` fue entrenado principalmente en inglés, por lo que la representación vectorial de una pregunta en español y su respuesta en inglés no son tan cercanas como deberían.

```
Pregunta (ES): "¿Qué excepción se genera al dividir entre cero?"
Chunk relevante (EN): "ZeroDivisionError: division by zero"
→ Distancia L2 más alta de lo esperado → puede que no llegue al top-5
```

Esto hace que FAISS recupere fragmentos menos relevantes, y el LLM genere una respuesta pobre o diga "no está en el contexto".

---

### 6. Cómo corregirlo

| Problema | Solución | Cambio en código |
| --- | --- | --- |
| Gap español↔inglés en embeddings | Modelo multilingüe | `SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")` |
| Prompt demasiado restrictivo | Permitir conocimiento propio si el contexto es débil | Cambiar `"ÚNICAMENTE el contexto"` → `"Prioriza el contexto; si no es suficiente, usa tu conocimiento"` |
| Chunks con contexto mezclado | Chunking semántico por párrafo/sección | Dividir por `\n\n` (párrafos) en lugar de ventana fija de palabras |
| Retrieval por distancia L2 simple | Hybrid search: BM25 + FAISS | Combinar similitud semántica con coincidencia de palabras clave |
| Chunks ruidosos por PDF infografía | Extracción multimodal | Convertir páginas a imagen y procesarlas con un LLM con visión |

> **Conclusión clave:** RAG es la arquitectura correcta para documentos privados o información que el LLM no conoce. Para documentos de conocimiento general, el beneficio es marginal y depende críticamente de la calidad de la extracción y del modelo de embeddings elegido.


# **Fin de la actividad chatbot: LLM + RAG**